# Download and archive Starlink / Amazon Kuiper / OneWeb

Save raw TLE and complete OMM JSON files with UTC timestamps. No synthetic TLEs are generated. Traditional TLEs cannot represent six-digit NORAD IDs, so downstream notebooks prefer JSON. The snapshot manifest records download URLs, hashes, counts, epoch ranges, and missing IDs.

First install the dependencies in `requirements.txt`; see `README.md`.
If the raw TLE request fails but JSON succeeds, save a separate TLE conversion for five-digit IDs, explicitly labeled `derived_from_omm`. Six-digit IDs remain available only in OMM.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

# Locate the shared module from either the repository or notebook directory.
ROOT = next((p for base in [Path.cwd(), *Path.cwd().parents]
             for p in [base, base / "generate satellite"]
             if (p / "satellite_pipeline.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the WorldNTN repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from satellite_pipeline import (
    download_snapshot, initialize_scenario, read_scenario, load_snapshot,
    export_tracks, preview_export, merge_exports, stamp,
)
# If dependencies are missing, run this in a separate notebook cell:
# %pip install -r "{ROOT / 'requirements.txt'}"
print("Workspace:", ROOT)


## Download settings

`auto`: reuse a snapshot from the last two hours; `refresh`: download a new snapshot; `local`: work entirely offline. Each format is requested once per constellation, without a retry loop.


In [ ]:
MODE = "auto"
RESET_START_TIME = False  # True starts a new common simulation epoch.
snapshot = download_snapshot(ROOT, mode=MODE)
print(snapshot)


In [ ]:
manifest = json.loads(snapshot.read_text())
rows = []
for constellation, entry in manifest["constellations"].items():
    for fmt in ("json", "tle", "derived_tle"):
        if fmt not in entry:
            continue
        rows.append({"constellation": constellation, "format": fmt, **entry[fmt]})
display(pd.DataFrame(rows))
for constellation, entry in manifest["constellations"].items():
    print(constellation, "OMM IDs absent from TLE:", len(entry["omm_ids_absent_from_tle"]))


## Fix the shared timeline and input snapshot

Write the current UTC time to `scenario.json` on the first run; later runs preserve it. All real and synthetic notebooks read this file. After changing the observer, timeline, or thresholds, regenerate every CSV that will be merged.

Defaults: the reference location at 39.95697°N, −105.16033°E, 1660 m; 24 hours at 120-second intervals; 720 samples; minimum elevation 25°.


In [ ]:
scenario = initialize_scenario(snapshot, ROOT, reset_start=RESET_START_TIME)
display(scenario)


## Approved orbital parameters

`orbit_info/` sits alongside `tle/` and contains official PDFs, source records, and a machine-readable parameter table. See `orbit_info/SOURCES.md` for the approval scope and simulation assumptions.


In [ ]:
display(pd.read_csv(ROOT / "orbit_info" / "approved_shells.csv"))
